# Getis Ord Gi*
Getis  and Ord's Gi and Gi* statistics are popular statistical approaches for finding statistically significant hot and cold spots across space. It compares the value of some numerical variable of a spatial record with those of the neighboring records. The nature of these neighborhoods is controlled by the user. 

In this example, we will use the Gi* statistic on the Kings County Homes dataset we prepared last week to identify regions of high and lower "density".

In [1]:
import wkls

# region = wkls.us.wa.kirkland.wkt()
neighbor_search_radius_degrees = .005  # ~400m
h3_zoom_level = 10  # coarser grid, ~150m edge length

# Small bounding box in central Kirkland (~0.5km²)
region = "POLYGON((-122.212 47.676, -122.200 47.676, -122.200 47.682, -122.212 47.682, -122.212 47.676))"

In [2]:
from sedona.spark import *
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

database = 'gde_gold'

sedona.sql(f'CREATE DATABASE IF NOT EXISTS org_catalog.{database}')
database

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/03/12 13:47:45 INFO core/src/lib.rs: Sedona native acceleration engine v0.12.0 ready
26/03/12 13:47:51 WARN ExecutorPodsLifecycleManager: 1 new failed executors.    
26/03/12 13:47:51 ERROR TaskSchedulerImpl: Lost executor 6 on 10.1.73.18: 
The executor with id 6 exited with exit code 1(generic, look at logs to clarify).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: registry.depot.dev/8jjbrshl5f:wherobots-spark-v2.1.13-db-22841738946
	 container state: terminated
	 container started at: 2026-03-12T13:47:48Z
	 container finished at: 2026-03-12T13:47:50Z
	 exit code: 1
	 termination reason: Error
      


'gde_gold'

## Spark Initialization
We will use Spark to run the Gi* algorithm. We initialize a Spark session with Sedona.


In [3]:
# from sedona.spark import *

# config = SedonaContext.builder().getOrCreate()
# sedona = SedonaContext.create(config)

## Filtering and Aggregation
In this notebook we assign an H3 cell to each record and filter down to only the region of interest. We aggregate the places data by the cell idenitier and find the number of places in each cell.


In [4]:
import pyspark.sql.functions as f
places_df = (
    sedona.table("org_catalog.gde_bronze.king_co_homes_conflated")
        .select(f.col("point"), f.col("sale_price"), f.col("sale_date"), f.col("sale_id"))
        .withColumn("h3Cell", ST_H3CellIDs(f.col("point"), h3_zoom_level, False)[0])
)

if region is not None:
    places_df = places_df.filter(ST_Intersects(ST_GeomFromText(f.lit(region)), f.col("geometry"))).repartition(100)

places_df.count()

376

In [5]:
hexes_df = (
    places_df
        .groupBy(f.col("h3Cell"))
        .agg(f.count("*").alias("num_places")) # how many places in this cell
        .withColumn("geometry", ST_H3ToGeom(f.array(f.col("h3Cell")))[0])
)

In [6]:
hexes_df.select(
    f.count("*").alias("total_cells"),
    f.min("num_places").alias("min"),
    f.max("num_places").alias("max"),
    f.avg("num_places").alias("mean")
).show()

[Stage 9:=============================>                             (1 + 1) / 2]

+-----------+---+---+------------------+
|total_cells|min|max|              mean|
+-----------+---+---+------------------+
|         34|  1| 33|11.058823529411764|
+-----------+---+---+------------------+



In [7]:
hexes_df = (
    places_df
        .groupBy(f.col("h3Cell"))
        .agg(
            f.count("*").alias("num_places"),
            f.sum("sale_price").alias("total_sale_value"),
            f.avg("sale_price").alias("avg_sale_price")
        )
        .withColumn("geometry", ST_H3ToGeom(f.array(f.col("h3Cell")))[0])
)

## Sanity Check our Variable
We want to make sure we have a good distribution of values in our variable that we will analyze. Specifically we are ensuring that our cells are not too small which would be indicated by the places counts all being very low. We generate deciles here to make sure that there is some good range of these values. An extreme negative example would be if these values were all zero and one.


In [8]:
hexes_df.select(f.percentile_approx("num_places", [x / 10.0 for x in range(11)])).collect()[0][0]

[1, 1, 3, 5, 6, 7, 13, 16, 19, 26, 33]

## Generate our Gi* statistic

Finally, we generate our statistic. There are a lot of variables to fine tune here; these are explained in the API documentation. Here we use the most typical parameters. The exception is the search radius which is always domain specific.

The output here will show us, among other things, a Z score and P value. A Z score shows how many standard deviations from the mean of the neighborhood the value is and the P score tells us the chance that value is from random variation rather than an actual phenomenon.



In [9]:
from sedona.spark import *


gi_df = g_local(
    add_binary_distance_band_column(
        hexes_df,
        neighbor_search_radius_degrees,
        include_self=True,
    ),
    "num_places",
    "weights",
    star=True
).cache()

gi_df.orderBy(f.col("P").asc()).show(5)

[Stage 65:==================================================> (1465 + 4) / 1500]

+------------------+----------+----------------+-----------------+--------------------+--------------------+------------------+------------------+--------------------+------------------+--------------------+
|            h3Cell|num_places|total_sale_value|   avg_sale_price|            geometry|             weights|                 G|                EG|                  VG|                 Z|                   P|
+------------------+----------+----------------+-----------------+--------------------+--------------------+------------------+------------------+--------------------+------------------+--------------------+
|622215090734727167|         8|         9340300|        1167537.5|POLYGON ((-122.20...|[{{62221509073351...|0.9813829787234043|0.8823529411764706|0.002043991728766...|2.1904199015528687|0.014246898994894286|
|622215090736300031|        16|        17118800|        1069925.0|POLYGON ((-122.20...|[{{62221509073351...|0.7313829787234043|0.5882352941176471|0.004769314033788...|2

## Visualize
Now we plot our statistics in Kepler. Once Kepler is rendered, you can color the cells by Z score and set the number of bands to 10 with the color palette that goes from blue to red. the bluest are the cold spots and reddest hottest.

In [10]:
from sedona.spark import *

kmap = SedonaKepler.create_map(places_df, "places")

SedonaKepler.add_df(
    kmap,
    gi_df.drop("weights"),
    "cells"
)

kmap

KeplerGl(data={'places': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, …

## Save out the statistics dataframe to a Gold table

In [11]:
%%time

gi_df.writeTo(f"org_catalog.{database}.king_co_homes_hotspots").createOrReplace()

CPU times: user 13.7 ms, sys: 2.4 ms, total: 16.1 ms
Wall time: 6.06 s


In [12]:
# Confirming table has rows
sedona.sql("SELECT COUNT(*) FROM org_catalog.gde_gold.analytics_gold").show()

+--------+
|count(1)|
+--------+
|     376|
+--------+

